# Claude Dataset: Cleaning & Analysis

This notebook cleans the synthetic Ferritic/Martensitic steel dataset generated by
Claude, then checks it against known metallurgical trends.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid", context="paper", font="sans-serif")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'sans-serif'
})


## 1. Cleaning the data

Claude's raw output was missing an explicit `Alloy_Type` column, but the sample IDs
(`F001`, `M002`, ...) implicitly encoded it, so it was engineered from the ID prefix.
It also included a redundant `Phase_Austenite_%` column that was zero for every row
(expected, since this is a Ferritic/Martensitic system with no austenite), and used
mixed Rockwell B/C hardness scales that needed converting to a common Vickers scale.


In [ ]:
def clean_claude_dataset(filepath):
    """
    Cleans the Claude dataset.

    - Drops the 'Phase_Austenite_%%' column (all zeros, not relevant here)
    - Engineers 'Alloy_Type' from the sample ID prefix (F = Ferritic, M = Martensitic)
    - Converts mixed HRB/HRC hardness scales to a single HV column
    - Fills a missing ferrite fraction from 100 - martensite fraction where possible
    - Fixes unit errors in elastic modulus and removes negative values
    - Imputes remaining gaps with the column median
    """
    if not os.path.exists(filepath):
        print(f"Error: File '{filepath}' not found.")
        return None

    df = pd.read_csv(filepath)
    print(f"Processing '{filepath}' | Initial shape: {df.shape}")

    # drop the all-zero austenite column
    austenite_cols = [col for col in df.columns if 'Austenite' in col]
    if austenite_cols:
        print(f"  - Dropping zero-value column(s): {austenite_cols}")
        df.drop(columns=austenite_cols, inplace=True)

    rename_map = {
        'Sample_ID': 'ID',
        'Phase_Ferrite_%': 'Ferrite_Vol_%',
        'Phase_Martensite_%': 'Martensite_Vol_%',
        'Yield_Strength_MPa': 'YS_MPa',
        'Ultimate_Tensile_Strength_MPa': 'UTS_MPa',
        'Elastic_Modulus_GPa': 'Modulus_GPa',
        'Elongation_%': 'Elongation_%',
        'Hardness_Value': 'Hardness_Value',
        'Hardness_Scale': 'Hardness_Scale',
        'Annealing_Temp_C': 'Heat_Treat_Temp_C',
        'Annealing_Time_h': 'Heat_Treat_Time_h'
    }
    df.rename(columns=rename_map, inplace=True)

    # engineer Alloy_Type from the ID prefix, since it isn't given directly
    if 'ID' in df.columns and 'Alloy_Type' not in df.columns:
        print("  - 'Alloy_Type' missing. Extracting from ID (F=Ferritic, M=Martensitic)...")
        df['Alloy_Type'] = df['ID'].astype(str).str[0].map({
            'F': 'Ferritic',
            'M': 'Martensitic'
        })
        if df['Alloy_Type'].isnull().any():
            df['Alloy_Type'].fillna(df['Alloy_Type'].mode()[0], inplace=True)

    # hardness: HRB (softer scale) and HRC (harder scale) both need converting to HV
    if 'Hardness_Value' in df.columns and 'Hardness_Scale' in df.columns:
        df['Hardness_Scale'] = df['Hardness_Scale'].astype(str).str.upper().str.strip()

        def normalize_hardness_claude(row):
            val = row['Hardness_Value']
            scale = row['Hardness_Scale']
            if pd.isna(val):
                return np.nan
            if scale == 'HRC':
                return (9.0 * val) + 55.0
            elif scale == 'HRB':
                # linear approximation, valid for HRB 70-100: HV ~= 3.4 * HRB - 113
                return (3.4 * val) - 113.0
            return val

        df['Hardness_HV'] = df.apply(normalize_hardness_claude, axis=1)
        df.drop(columns=['Hardness_Value', 'Hardness_Scale'], inplace=True)
        print("  - Standardized hardness to a single 'Hardness_HV' column.")

    # unit fix: modulus values over 1000 are in MPa, not GPa
    if 'Modulus_GPa' in df.columns:
        mask_units = df['Modulus_GPa'] > 1000
        if mask_units.any():
            df.loc[mask_units, 'Modulus_GPa'] = df.loc[mask_units, 'Modulus_GPa'] / 1000.0

    # remove physically impossible negative values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in [c for c in numeric_cols if 'Temp' not in c]:
        df.loc[df[col] < 0, col] = np.nan

    # if ferrite fraction is missing but martensite is known, derive it from the balance
    if 'Ferrite_Vol_%' in df.columns and 'Martensite_Vol_%' in df.columns:
        mask_f_miss = df['Ferrite_Vol_%'].isnull() & df['Martensite_Vol_%'].notnull()
        df.loc[mask_f_miss, 'Ferrite_Vol_%'] = 100 - df.loc[mask_f_miss, 'Martensite_Vol_%']

    # fill remaining gaps
    for col in numeric_cols:
        if col in df.columns and df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    df['Source'] = 'Claude'
    return df


filename = '../data/raw/claude.csv'
if os.path.exists(filename):
    claude_clean = clean_claude_dataset(filename)
    if claude_clean is not None:
        claude_clean.to_csv('../data/cleaned/cleaned_claude.csv', index=False)
        print("\nSaved cleaned data to 'cleaned_claude.csv'.")
        print(claude_clean[['ID', 'Alloy_Type']].head(4))
        print(claude_clean['Hardness_HV'].describe().round(1))


## 2. Summary statistics

In addition to overall statistics, properties are also broken down by alloy type here,
since Ferritic and Martensitic steels are expected to have quite different property
ranges — a model that generates realistic data should reproduce that split.


In [ ]:
filename = '../data/cleaned/cleaned_claude.csv'
if os.path.exists(filename):
    df = pd.read_csv(filename)

    target_cols = [
        'YS_MPa', 'UTS_MPa', 'Elongation_%',
        'Hardness_HV', 'Ferrite_Vol_%', 'Modulus_GPa'
    ]
    existing_cols = [c for c in target_cols if c in df.columns]

    print("--- Global statistics (all samples) ---")
    stats_df = df[existing_cols]
    summary_stats = stats_df.describe().T
    summary_stats['Median'] = stats_df.median()
    summary_stats['Mode'] = stats_df.mode().iloc[0]
    summary_stats['Skewness'] = stats_df.skew()
    final_stats = summary_stats[['mean', 'Median', 'Mode', 'std', 'min', 'max', 'Skewness']]
    print(final_stats.round(2))
    final_stats.round(2).to_csv('../data/cleaned/claude_statistics_summary.csv')

    if 'Alloy_Type' in df.columns:
        print("\n--- Statistics by alloy type ---")
        grouped = df.groupby('Alloy_Type')[existing_cols].mean().round(2)
        print(grouped)
        grouped.to_csv('../data/cleaned/claude_statistics_by_alloy.csv')
else:
    print(f"'{filename}' not found. Run the cleaning cell first.")


## 3. Exploratory plots


In [ ]:
filename = '../data/cleaned/cleaned_claude.csv'

if os.path.exists(filename):
    df = pd.read_csv(filename)

    sns.set_theme(style="whitegrid", context="paper", font="sans-serif", font_scale=1.2)
    custom_palette = {"Ferritic": "#2b8cbe", "Martensitic": "#d73027"}
    unique_classes = df['Alloy_Type'].unique()
    for cls in unique_classes:
        if cls not in custom_palette:
            custom_palette[cls] = "#95a5a6"

    numeric_df = df.select_dtypes(include=[np.number])
    if 'ID' in numeric_df.columns:
        numeric_df = numeric_df.drop(columns=['ID'])

    plt.figure(figsize=(14, 12))
    sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm",
                center=0, square=True, linewidths=.5, cbar_kws={"shrink": .8})
    plt.title('Correlation Matrix of All Variables', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

    if 'Grain_Size_um' in df.columns:
        plt.figure(figsize=(9, 7))
        sns.scatterplot(data=df, x='Grain_Size_um', y='YS_MPa', hue='Alloy_Type',
                        palette=custom_palette, s=120, edgecolor='black', alpha=0.8)
        sns.regplot(data=df, x='Grain_Size_um', y='YS_MPa', scatter=False,
                    order=2, color='#34495e', line_kws={'linestyle':'--'})
        plt.title('Yield Strength vs. Grain Size (Hall-Petch)', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Grain Size (\u00b5m)', fontsize=14, fontweight='bold')
        plt.ylabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
        plt.legend(title='Alloy Type')
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(9, 7))
    sns.scatterplot(data=df, x='YS_MPa', y='Elongation_%', hue='Alloy_Type',
                    palette=custom_palette, style='Alloy_Type', s=120, alpha=0.9)
    plt.title('Yield Strength vs. Elongation', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
    plt.ylabel('Elongation (%)', fontsize=14, fontweight='bold')
    plt.legend(title='Alloy Type')
    plt.tight_layout()
    plt.show()

    if 'C_wt%' in df.columns:
        plt.figure(figsize=(9, 7))
        sns.scatterplot(data=df, x='C_wt%', y='Hardness_HV', hue='Alloy_Type',
                        palette=custom_palette, s=120, edgecolor='black')
        plt.title('Hardness vs. Carbon Content', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Carbon Content (wt%)', fontsize=14, fontweight='bold')
        plt.ylabel('Hardness (HV)', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(9, 7))
    sns.regplot(data=df, x='Hardness_HV', y='UTS_MPa',
                scatter_kws={'s': 80, 'alpha': 0.6, 'color':'purple'},
                line_kws={'color':'black', 'linestyle':'--'})
    plt.title('UTS vs. Hardness', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Hardness (HV)', fontsize=14, fontweight='bold')
    plt.ylabel('Ultimate Tensile Strength (MPa)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    if 'Ferrite_Vol_%' in df.columns and 'C_wt%' in df.columns:
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=df, x='Ferrite_Vol_%', y='YS_MPa', hue='Alloy_Type',
                        size='C_wt%', sizes=(50, 300), palette=custom_palette,
                        alpha=0.7, edgecolor='black')
        plt.title('Yield Strength vs. Ferrite Fraction', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Ferrite Volume Fraction (%)', fontsize=14, fontweight='bold')
        plt.ylabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
        plt.gca().invert_xaxis()
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Alloy & Carbon %")
        plt.tight_layout()
        plt.show()

    if 'Heat_Treat_Temp_C' in df.columns:
        plt.figure(figsize=(9, 7))
        df['Process_Label'] = df['Alloy_Type'].map({
            'Ferritic': 'Ferritic (Annealed)',
            'Martensitic': 'Martensitic (Quenched)'
        }).fillna(df['Alloy_Type'])

        sns.scatterplot(data=df, x='Heat_Treat_Temp_C', y='YS_MPa', hue='Process_Label',
                        palette={'Ferritic (Annealed)': '#2b8cbe', 'Martensitic (Quenched)': '#d73027'},
                        s=100, alpha=0.7, legend=False)
        sns.lineplot(data=df, x='Heat_Treat_Temp_C', y='YS_MPa', hue='Process_Label',
                     palette={'Ferritic (Annealed)': '#2b8cbe', 'Martensitic (Quenched)': '#d73027'},
                     marker=None, linewidth=3, errorbar=None)

        plt.title('Yield Strength vs. Heat Treatment Temperature', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Heat Treatment Temperature (\u00b0C)', fontsize=14, fontweight='bold')
        plt.ylabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
        plt.legend(title='Alloy Condition')
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(9, 7))
    sns.violinplot(data=df, x='Alloy_Type', y='YS_MPa', hue='Alloy_Type',
                   palette=custom_palette, inner="quartile", dodge=False)
    plt.title('Yield Strength Distribution by Alloy', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Alloy Type', fontsize=14, fontweight='bold')
    plt.ylabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
    plt.legend([],[], frameon=False)
    plt.tight_layout()
    plt.show()

else:
    print(f"'{filename}' not found.")


## Observations

Claude's data showed the strongest adherence to real metallurgical relationships of the
four models tested. The correlation matrix showed physically sensible relationships
(hardness positively correlated with carbon content, yield strength negatively
correlated with elongation). The Hall-Petch trend was the clearest of all four, with
two well-separated clusters for fine-grained martensite (high strength) and coarser
ferrite (lower strength). The processing-property link was also captured correctly:
higher annealing temperatures were associated with softening in the ferritic samples.
